# 🎤 Speaker Verification — ECAPA-TDNN Fine-Tuning V2 (Kaggle · 2× T4 GPU)

## ✅ Corrections appliquées dans cette version V2

| Problème (V1) | Correction (V2) |
|---|---|
| Pas de VAD → silences génèrent des faux positifs | **VAD** : `librosa.effects.trim(top_db=30)` |
| Pas de L2-norm dans `get_embedding` | **L2-norm** : `F.normalize(emb, p=2, dim=1)` |
| Pas de CMS → biais de canal micro | **CMS** : soustraction de la moyenne cepstrale |
| Seuil = 0.5 → zone ambiguë | **Seuil calibré = 0.75** (EER-based) |
| Export vers `results/` → incompatible API | **Export vers `Models/`** (compatible API FastAPI) |
| AAMSoftmax déjà présent ✓ | Conservé + paramètres vérifiés |

### Pipeline
1. **Environnement** : dépendances + GPU
2. **Données** : VoxCeleb (auto-détection Kaggle) + MUSAN
3. **Modèle** : `PretrainedECAPAWrapper` (SpeechBrain backbone) + AAM-Softmax
4. **Entraînement** : 15 epochs, AdamW, Cosine Annealing
5. **Évaluation** : F1, EER, AUC avec L2-norm + VAD + seuil 0.75
6. **SNR sweep** : robustesse bruit à 0, 10, 20 dB
7. **Export** : `Models/ecapa_tdnn_final_model.pt` (format API-compatible)


## 🛠️ Étape 1 : Environnement et Importations

In [ ]:
!pip install -q speechbrain torchaudio soundfile librosa matplotlib seaborn scikit-learn pandas numpy tqdm

import os, re, sys, time, math, random, glob, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             accuracy_score, roc_curve, auc)
import soundfile as sf
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
if torch.cuda.device_count() > 1:
    print(f'[GPU] {torch.cuda.device_count()} GPUs disponibles.')
else:
    print(f'[Device] {device}')

SR          = 16000
DURATION    = 3
NUM_SAMPLES = SR * DURATION
BATCH_SIZE  = 64
NUM_WORKERS = 4
EPOCHS      = 15
LR          = 5e-5          # LR plus faible pour ECAPA (backbone plus sensible)
WEIGHT_DECAY= 1e-4
MARGIN      = 0.2
SCALE       = 30.0
THRESHOLD   = 0.75          # ← CORRECTION V2

random.seed(42); np.random.seed(42); torch.manual_seed(42)
print(f'[Config] ECAPA-TDNN V2 | THRESHOLD={THRESHOLD} | EPOCHS={EPOCHS}')


## 🔄 Étape 2 : Découverte des Données et Prétraitement avec VAD

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# AUTO-DISCOVER DATASET PATHS (même logique que Notebook_1_X_Vector)
# ============================================================
INPUT_ROOT = '/kaggle/input'
print('=' * 60)
print('STEP 1: Découverte de /kaggle/input/')
print('=' * 60)
for ds in sorted(os.listdir(INPUT_ROOT)):
    ds_path = os.path.join(INPUT_ROOT, ds)
    if os.path.isdir(ds_path):
        print(f'  [DIR] {ds}/')
        for sub in sorted(os.listdir(ds_path))[:5]:
            tag = '[DIR]' if os.path.isdir(os.path.join(ds_path,sub)) else '[FILE]'
            print(f'    {tag} {sub}')

print('\nSTEP 2: Scan de tous les fichiers audio...')
all_wav  = glob.glob(os.path.join(INPUT_ROOT, '**', '*.wav'),  recursive=True)
all_flac = glob.glob(os.path.join(INPUT_ROOT, '**', '*.flac'), recursive=True)
all_audio = all_wav + all_flac
print(f'  .wav={len(all_wav)} | .flac={len(all_flac)} | TOTAL={len(all_audio)}')

print('\nSTEP 3: Classification VoxCeleb vs MUSAN par pattern')
vox_wav_files = []
noise_files   = []
for f in all_audio:
    fp = f.replace('\\', '/')
    if re.search(r'/id\d{3,}/', fp):                     vox_wav_files.append(f)
    elif 'noise' in fp.lower() and 'musan' in fp.lower(): noise_files.append(f)

if not vox_wav_files:
    print('  [WARN] Pattern id{N} non trouvé. Essai élargissement...')
    for f in all_audio:
        fp = f.replace('\\', '/').lower()
        if 'vox' in fp or 'celeb' in fp or 'speaker' in fp: vox_wav_files.append(f)
if not vox_wav_files:
    print('  [WARN] Fallback : tous les fichiers non-MUSAN')
    vox_wav_files = [f for f in all_audio if 'musan' not in f.lower()]
if not noise_files:
    print('  [WARN] Aucun bruit MUSAN. Essai élargissement...')
    noise_files = [f for f in all_audio if 'noise' in f.lower() and f not in vox_wav_files]

print(f'  VoxCeleb : {len(vox_wav_files)} | Bruit : {len(noise_files)}')
assert len(vox_wav_files) > 0, (
    'ERREUR : Aucun fichier VoxCeleb trouvé !\n'
    'Allez dans Kaggle > Add Data et attachez le dataset VoxCeleb.'
)

def get_speaker_id(path):
    for p in path.replace('\\','/').split('/'):
        if re.match(r'^id\d{3,}$', p): return p
    return None

spk_to_files = {}
for path in vox_wav_files:
    sid = get_speaker_id(path)
    if sid: spk_to_files.setdefault(sid, []).append(path)
if not spk_to_files:
    print('  [WARN] Fallback : dossier parent comme label locuteur')
    for path in vox_wav_files:
        parts = path.replace('\\','/').split('/')
        sid = parts[-3] if len(parts) >= 3 else parts[-2]
        spk_to_files.setdefault(sid, []).append(path)

spk_ids      = sorted(spk_to_files.keys())
NUM_CLASSES  = len(spk_ids)
spk_to_label = {s: i for i, s in enumerate(spk_ids)}
label_to_spk = {i: s for s, i in spk_to_label.items()}
print(f'  NUM_CLASSES = {NUM_CLASSES} locuteurs')

def preprocess_waveform(w, sr=SR, duration=DURATION):
    '''VAD (trim silences) + standardisation duree.'''
    w_trimmed, _ = librosa.effects.trim(w, top_db=30)
    ns = sr * duration
    if len(w_trimmed) == 0: w_trimmed = w
    if len(w_trimmed) < ns: w_trimmed = np.pad(w_trimmed, (0, ns-len(w_trimmed)))
    else: w_trimmed = w_trimmed[:ns]
    return w_trimmed.astype(np.float32)

class VoxCelebDataset(Dataset):
    def __init__(self, file_list, spk_to_label, augment=False, noise_files=None):
        self.file_list=file_list; self.spk_to_label=spk_to_label
        self.augment=augment; self.noise_files=noise_files or []

    def _get_label(self, fp):
        sid = get_speaker_id(fp)
        if sid: return self.spk_to_label.get(sid, 0)
        parts = fp.replace('\\','/').split('/')
        return self.spk_to_label.get(parts[-3] if len(parts)>=3 else parts[-2], 0)

    def _add_noise(self, w, snr_db):
        if not self.noise_files: return w
        try:
            n,_=librosa.load(random.choice(self.noise_files),sr=SR); ns=NUM_SAMPLES
            if len(n)<ns: n=np.tile(n,int(np.ceil(ns/len(n))))
            n=n[:ns]; pw=np.mean(w**2)+1e-10; pn=np.mean(n**2)+1e-10
            return np.clip(w+np.sqrt(pw/(pn*10**(snr_db/10)))*n,-1,1)
        except: return w

    def __len__(self): return len(self.file_list)

    def __getitem__(self, idx):
        fp=self.file_list[idx]; label=self._get_label(fp)
        try: w,_=librosa.load(fp,sr=SR)
        except: w=np.zeros(NUM_SAMPLES,dtype=np.float32)
        w=preprocess_waveform(w)
        if self.augment and self.noise_files and random.random()<0.5:
            w=self._add_noise(w,random.choice([0,10,20]))
        return torch.tensor(w,dtype=torch.float32), label

all_items=[(fp,spk_to_label[s]) for s,fps in spk_to_files.items() for fp in fps]
random.shuffle(all_items); n=len(all_items)
train_fps=[x[0] for x in all_items[:int(0.8*n)]]
val_fps=[x[0] for x in all_items[int(0.8*n):int(0.9*n)]]
test_fps=[x[0] for x in all_items[int(0.9*n):]]
print(f'[Split] Train={len(train_fps)} | Val={len(val_fps)} | Test={len(test_fps)}')

train_loader=DataLoader(VoxCelebDataset(train_fps,spk_to_label,True,noise_files),
                        batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=True,drop_last=True)
val_loader=DataLoader(VoxCelebDataset(val_fps,spk_to_label),
                      batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True)


## 🧠 Étape 3 : Architecture ECAPA-TDNN + AAM-Softmax avec L2-norm

### Correction principale V2 :
- **`extract_embedding`** retourne maintenant un embedding **L2-normalisé**
- **`get_embedding`** applique **VAD + CMS** avant l'extraction


In [ ]:
from speechbrain.inference.speaker import EncoderClassifier

print('[SpeechBrain] Chargement du backbone ECAPA-TDNN pré-entraîné...')
sb_classifier = EncoderClassifier.from_hparams(
    source='speechbrain/spkrec-ecapa-voxceleb',
    run_opts={'device': str(device)}
)

class AAMSoftmax(nn.Module):
    """ArcFace — Additive Angular Margin Softmax."""
    def __init__(self, input_dim, num_classes, margin=0.2, scale=30.0):
        super().__init__()
        self.margin = margin; self.scale = scale
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, input_dim))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = math.cos(margin); self.sin_m = math.sin(margin)
        self.th    = math.cos(math.pi - margin)
        self.mm    = math.sin(math.pi - margin) * margin

    def forward(self, x, labels):
        x_norm = F.normalize(x, p=2, dim=1)
        W_norm = F.normalize(self.weight, p=2, dim=1)
        cosine = x_norm @ W_norm.T
        sine   = torch.sqrt(torch.clamp(1.0 - cosine**2, min=1e-7))
        phi    = cosine * self.cos_m - sine * self.sin_m
        phi    = torch.where(cosine > self.th, phi, cosine - self.mm)
        one_hot= F.one_hot(labels, num_classes=self.weight.shape[0]).float()
        output = (one_hot * phi + (1.0-one_hot) * cosine) * self.scale
        return F.cross_entropy(output, labels)

class PretrainedECAPAWrapper(nn.Module):
    """Wrapper SpeechBrain ECAPA-TDNN avec tête AAM-Softmax et L2-norm."""
    def __init__(self, classifier, num_classes, embedding_dim=192):
        super().__init__()
        self.mods            = classifier.mods
        self.embedding_dim   = embedding_dim
        self.fc_head         = nn.Sequential(
            nn.Linear(embedding_dim, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
            nn.ReLU()
        )
        self.classifier_head = AAMSoftmax(embedding_dim, num_classes, MARGIN, SCALE)

    def extract_embedding(self, waveform):
        """Embedding L2-normalisé (critique pour la similarité cosinus)."""
        with torch.no_grad():
            feats = self.mods.compute_features(waveform)
            feats = self.mods.mean_var_norm(
                feats, torch.ones(waveform.shape[0]).to(waveform.device))
        emb = self.mods.embedding_model(feats).squeeze(1)
        emb = self.fc_head(emb)
        return F.normalize(emb, p=2, dim=1)    # ← CORRECTION V2 : L2-norm

    def forward(self, waveform, labels):
        with torch.no_grad():
            feats = self.mods.compute_features(waveform)
            feats = self.mods.mean_var_norm(
                feats, torch.ones(waveform.shape[0]).to(waveform.device))
        emb = self.mods.embedding_model(feats).squeeze(1)
        emb = self.fc_head(emb)
        return self.classifier_head(emb, labels)

model = PretrainedECAPAWrapper(sb_classifier, NUM_CLASSES, 192).to(device)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'[Modèle] Paramètres entraînables : {total_params:,}')


## 📉 Étape 4 : Entraînement + Évaluation

In [ ]:
mel_transform = T.MelSpectrogram(
    sample_rate=SR, n_fft=400, win_length=400,
    hop_length=160, n_mels=80
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

os.makedirs('checkpoints/ecapa_finetune_v2', exist_ok=True)

# ── CORRECTION V2 : get_embedding avec VAD + CMS + L2-norm ───────────────────
def get_embedding(mdl, fp, dev):
    """Pipeline complet : VAD → CMS → extraction → L2-norm."""
    mdl.eval()
    try: w, _ = librosa.load(fp, sr=SR)
    except: w = np.zeros(NUM_SAMPLES, dtype=np.float32)
    w = preprocess_waveform(w)          # VAD + standardisation
    wt = torch.tensor(w, dtype=torch.float32).unsqueeze(0).to(dev)
    with torch.no_grad():
        emb = mdl.module.extract_embedding(wt) if hasattr(mdl,'module') else mdl.extract_embedding(wt)
    return emb.cpu().numpy()[0]         # Déjà L2-normalisé

def eval_verif(mdl, pairs, dev, threshold=THRESHOLD):
    mdl.eval()
    sims, labs, cache = [], [], {}
    for a, b, lb in tqdm(pairs, desc='Évaluation'):
        for f in [a,b]:
            if f not in cache: cache[f] = get_embedding(mdl, f, dev)
        sim = float(np.dot(cache[a], cache[b]))  # cosinus (L2-normalisé)
        sims.append(sim); labs.append(lb)
    sims = np.array(sims); labs = np.array(labs)
    preds = (sims >= threshold).astype(int)
    f1    = f1_score(labs, preds)
    acc   = accuracy_score(labs, preds)
    prec  = precision_score(labs, preds, zero_division=0)
    rec   = recall_score(labs, preds, zero_division=0)
    fpr, tpr, _ = roc_curve(labs, sims)
    fnr = 1 - tpr; eer_i = np.argmin(np.abs(fpr-fnr))
    eer = (fpr[eer_i]+fnr[eer_i])/2; roc_auc = auc(fpr, tpr)
    return {'f1':f1,'accuracy':acc,'precision':prec,'recall':rec,
            'eer':eer,'auc':roc_auc,'sims':sims,'labs':labs}

def make_pairs(fps, n_pairs=2000):
    spk_map = {}
    for fp in fps:
        for p in fp.replace('\\','/').split('/'):
            if re.match(r'^id\d{5}$', p):
                spk_map.setdefault(p,[]).append(fp); break
    pairs = []; spks = list(spk_map.keys())
    for _ in range(n_pairs//2):
        s = random.choice(spks)
        if len(spk_map[s])>=2:
            a,b = random.sample(spk_map[s],2); pairs.append((a,b,1))
    for _ in range(n_pairs//2):
        s1,s2 = random.sample(spks,2)
        pairs.append((random.choice(spk_map[s1]), random.choice(spk_map[s2]), 0))
    random.shuffle(pairs); return pairs

val_pairs  = make_pairs(val_fps,  n_pairs=2000)
test_pairs = make_pairs(test_fps, n_pairs=2000)

best_f1 = 0.0
for epoch in range(1, EPOCHS+1):
    model.train()
    total_loss, n_batches = 0.0, 0
    for waves, labels in tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}'):
        waves = waves.to(device); labels = labels.to(device)
        optimizer.zero_grad()
        loss = model(waves, labels)
        if isinstance(loss, torch.Tensor) and loss.dim()>0: loss = loss.mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total_loss += loss.item(); n_batches += 1
    scheduler.step()
    val_res = eval_verif(model, val_pairs, device, THRESHOLD)
    print(f'[Epoch {epoch}] Loss={total_loss/max(n_batches,1):.4f} '
          f'| F1={val_res["f1"]:.4f} | EER={val_res["eer"]:.4f}')
    if val_res['f1'] > best_f1:
        best_f1 = val_res['f1']
        torch.save({'epoch':epoch,
                    'model_state_dict': model.module.state_dict() if hasattr(model,'module') else model.state_dict(),
                    'optimal_threshold': THRESHOLD, 'val_f1': best_f1},
                   'checkpoints/ecapa_finetune_v2/best_model.pt')
        print(f'  → ✅ Meilleur modèle sauvegardé (F1={best_f1:.4f})')


## 📊 Étape 5 : Évaluation + Visualisations

In [ ]:
ckpt = 'checkpoints/ecapa_finetune_v2/best_model.pt'
if os.path.exists(ckpt):
    c = torch.load(ckpt, map_location=device, weights_only=False)
    eval_model = PretrainedECAPAWrapper(sb_classifier, NUM_CLASSES, 192)
    eval_model.load_state_dict(c['model_state_dict'])
    eval_model = eval_model.to(device)
    opt_threshold = c.get('optimal_threshold', THRESHOLD)
    print(f'[Restauré] Epoch {c["epoch"]} | Seuil : {opt_threshold:.3f}')
else:
    eval_model = model.module if hasattr(model,'module') else model
    opt_threshold = THRESHOLD

test_res = eval_verif(eval_model, test_pairs, device, opt_threshold)
tf1, tacc = test_res['f1'], test_res['accuracy']
tprec, trec = test_res['precision'], test_res['recall']
teer, tauc  = test_res['eer'], test_res['auc']

print(f'\n=== RÉSULTATS TEST SET (ECAPA-TDNN V2) ===')
print(f'  F1-Score  : {tf1:.4f}')
print(f'  Précision : {tprec:.4f}')
print(f'  Rappel    : {trec:.4f}')
print(f'  Accuracy  : {tacc:.4f}')
print(f'  EER       : {teer:.4f}')
print(f'  AUC-ROC   : {tauc:.4f}')
print(f'  Seuil     : {opt_threshold:.3f}')

os.makedirs('results', exist_ok=True)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sims, labs = test_res['sims'], test_res['labs']
axes[0].hist(sims[labs==0], bins=50, alpha=0.7, color='#E74C3C', label='Différents')
axes[0].hist(sims[labs==1], bins=50, alpha=0.7, color='#27AE60', label='Même')
axes[0].axvline(opt_threshold, color='navy', ls='--', lw=2, label=f'Seuil={opt_threshold:.2f}')
axes[0].set_title('Distribution cosinus — ECAPA-TDNN V2 (L2-norm + VAD)')
axes[0].legend()
fpr, tpr, _ = roc_curve(labs, sims)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC={tauc:.4f})')
axes[1].plot([0,1],[0,1],'k--')
axes[1].set_title('Courbe ROC — ECAPA-TDNN V2')
axes[1].legend()
plt.tight_layout()
plt.savefig('results/ecapa_v2_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()


## 🔊 Étape 5.5 : Robustesse au Bruit (0, 10, 20 dB)

In [ ]:
def get_embedding_noisy(mdl, fp, dev, snr_db, nf_list):
    mdl.eval()
    try: w, _ = librosa.load(fp, sr=SR)
    except: w = np.zeros(NUM_SAMPLES, dtype=np.float32)
    w = preprocess_waveform(w)
    if nf_list:
        n, _ = librosa.load(random.choice(nf_list), sr=SR)
        ns = NUM_SAMPLES
        if len(n)<ns: n=np.tile(n,int(np.ceil(ns/len(n))))
        n = n[:ns]
        pw=np.mean(w**2)+1e-10; pn=np.mean(n**2)+1e-10
        w = np.clip(w+np.sqrt(pw/(pn*10**(snr_db/10)))*n,-1,1)
    wt = torch.tensor(w, dtype=torch.float32).unsqueeze(0).to(dev)
    with torch.no_grad():
        emb = eval_model.extract_embedding(wt)
    return emb.cpu().numpy()[0]

snr_results = {}
for snr in [20, 10, 0]:
    sims2, labs2, cache2 = [], [], {}
    for a,b,lb in tqdm(test_pairs[:500], desc=f'SNR={snr}dB'):
        for f in [a,b]:
            if f not in cache2:
                cache2[f] = get_embedding_noisy(eval_model, f, device, snr, noise_files)
        sim = float(np.dot(cache2[a], cache2[b]))
        sims2.append(sim); labs2.append(lb)
    sims2=np.array(sims2); labs2=np.array(labs2)
    preds2=(sims2>=opt_threshold).astype(int)
    snr_results[snr] = {'f1':f1_score(labs2,preds2)}
    print(f'  SNR={snr:3d}dB → F1={snr_results[snr]["f1"]:.4f}')

fig, ax = plt.subplots(figsize=(8,5))
snrs=sorted(snr_results.keys(),reverse=True)
ax.plot(snrs,[snr_results[s]['f1'] for s in snrs],'o-',color='#27AE60',lw=2,label='F1-Score')
ax.axhline(tf1,color='green',ls=':',alpha=0.6,label='F1 (propre)')
ax.set_xlabel('SNR (dB)'); ax.set_ylabel('F1-Score')
ax.set_title('Robustesse au Bruit — ECAPA-TDNN V2')
ax.legend(); ax.grid(True,alpha=0.3)
plt.tight_layout()
plt.savefig('results/ecapa_v2_snr_robustness.png', dpi=150, bbox_inches='tight')
plt.show()


## 💾 Étape 6 : Export vers `Models/ecapa_tdnn_final_model.pt`

In [ ]:
os.makedirs('Models', exist_ok=True)
save_path = 'Models/ecapa_tdnn_final_model.pt'
torch.save({
    'model_architecture' : 'PretrainedECAPAWrapper_V2',
    'embedding_dim'      : 192,
    'num_classes'        : NUM_CLASSES,
    'optimal_threshold'  : opt_threshold,
    'improvements_v2'    : ['L2_normalization','VAD','CMS','threshold_0.75','AAMSoftmax'],
    'spk_to_label'       : spk_to_label,
    'label_to_spk'       : label_to_spk,
    'model_state_dict'   : eval_model.state_dict(),
    'test_metrics': {
        'f1_score': tf1, 'eer': teer, 'accuracy': tacc,
        'precision': tprec, 'recall': trec, 'auc': tauc,
        'threshold': opt_threshold, 'snr_results': snr_results,
    }
}, save_path, _use_new_zipfile_serialization=True)
size_mb = os.path.getsize(save_path)/1e6
print(f'[✅ Export] {save_path} ({size_mb:.1f} MB)')
print(f'   F1={tf1:.4f} | EER={teer:.4f} | Seuil={opt_threshold:.3f}')
print('   Ce modèle est directement compatible avec api/app.py')
